# Carbon-Aware RecSys — Full Pipeline Run

Runs the complete pipeline on Colab GPU:
1. **Preprocess** — download raw data, predict PCF, build interim CSVs
2. **Train** — BPR, NeuMF, LightGCN × 3 categories
3. **Rerank** — λ sweep for each model/category
4. **Evaluate** — Pareto frontiers, summary tables, plots
5. **Paper plots** — generate figures for docs/main.tex

Results are saved to Google Drive so they persist across sessions.

In [ ]:
#@title ⚙️ Configuration

REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
BRANCH = 'test-run'

CATEGORIES = ['electronics', 'home_and_kitchen', 'sports_and_outdoors']
MODELS = ['BPR', 'NeuMF_pretrained', 'LightGCN']

# Set to True to re-run even if cached outputs exist
FORCE_TRAIN = False
FORCE_PREPROCESS = False

# Save results to Drive?
USE_DRIVE = True

In [ ]:
#@title 1. Mount Drive & clone repo
from pathlib import Path
import subprocess, sys, os, shutil

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/carbon-aware-recsys')
DRIVE_RESULTS = Path('/content/drive/MyDrive/carbon-aware-recsys-results')

# Clone or update repo
if not (PROJECT_ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(PROJECT_ROOT)
print(f'Repo: {PROJECT_ROOT}')
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
#@title 2. Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

# Verify critical imports
import torch, recbole
print(f'torch={torch.__version__}  CUDA={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')
print(f'recbole={recbole.__version__}')

In [ ]:
#@title 3. Restore cached results from Drive (if any)
if USE_DRIVE and DRIVE_RESULTS.exists():
    for subdir in ['output/results', 'output/models', 'data/interim', 'data/processed/carbon', 'data/raw']:
        src = DRIVE_RESULTS / subdir
        dst = PROJECT_ROOT / subdir
        if src.exists():
            dst.mkdir(parents=True, exist_ok=True)
            subprocess.run(['rsync', '-a', str(src) + '/', str(dst) + '/'], check=True)
            print(f'Restored {subdir}')
    print('Done restoring cached results.')
else:
    print('No cached results on Drive (or Drive disabled). Starting fresh.')

In [ ]:
#@title 4. Preprocess: download data + predict PCF + build interim CSVs
import time

interim_ready = all(
    (PROJECT_ROOT / 'data' / 'interim' / split / f'{cat}.csv').exists()
    for cat in CATEGORIES
    for split in ['train', 'val', 'test']
)

if interim_ready and not FORCE_PREPROCESS:
    print('Interim data already exists for all categories. Skipping preprocess.')
else:
    # Step A: predict carbon footprints (retrieval-only, no LLM)
    pcf_path = PROJECT_ROOT / 'data' / 'processed' / 'carbon' / 'amazon_pcf_predictions.csv'
    if not pcf_path.exists() or FORCE_PREPROCESS:
        print('Running PCF prediction (retrieval-only)...')
        t0 = time.time()
        subprocess.run([
            sys.executable, '-u', 'scripts/predict_carbon.py',
            '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
            '--num-threads', '4',
            '--skip-llm',
        ], check=True)
        print(f'PCF prediction done in {time.time()-t0:.0f}s')
    else:
        print('PCF predictions already cached.')

    # Step B: preprocess (download raw + merge meta + assign PCF -> interim CSVs)
    print('Running preprocessing...')
    t0 = time.time()
    subprocess.run([
        sys.executable, '-u', '-c',
        'from src.data.preprocess import merge_meta_with_interactions; merge_meta_with_interactions()'
    ], check=True)
    print(f'Preprocessing done in {time.time()-t0:.0f}s')

In [ ]:
#@title 5. Train all models
import time

for model in MODELS:
    for cat in CATEGORIES:
        scores_path = PROJECT_ROOT / 'output' / 'results' / f'{cat}_{model}_scores.parquet'
        if scores_path.exists() and not FORCE_TRAIN:
            print(f'[SKIP] {cat}/{model} - scores already exist')
            continue

        print(f'\n{"="*60}')
        print(f'Training {model} on {cat}...')
        print(f'{"="*60}', flush=True)
        t0 = time.time()

        cmd = [
            sys.executable, '-u', 'scripts/01_train_recommender.py',
            '--category', cat,
            '--model', model,
        ]
        if FORCE_TRAIN:
            cmd.append('--force-train')

        result = subprocess.run(cmd)
        elapsed = time.time() - t0

        if result.returncode != 0:
            print(f'[FAIL] {cat}/{model} after {elapsed:.0f}s (rc={result.returncode})')
        else:
            print(f'[DONE] {cat}/{model} in {elapsed:.0f}s')

In [ ]:
#@title 6. Rerank all models
for model in MODELS:
    for cat in CATEGORIES:
        metrics_path = PROJECT_ROOT / 'output' / 'results' / f'{cat}_{model}_reranking_metrics.json'
        if metrics_path.exists() and not FORCE_TRAIN:
            print(f'[SKIP] {cat}/{model} - reranking metrics already exist')
            continue

        print(f'Reranking {cat}/{model}...', flush=True)
        t0 = time.time()
        result = subprocess.run([
            sys.executable, '-u', 'scripts/02_rerank.py',
            '--category', cat,
            '--model', model,
        ])
        elapsed = time.time() - t0
        status = 'DONE' if result.returncode == 0 else 'FAIL'
        print(f'[{status}] {cat}/{model} rerank in {elapsed:.0f}s')

In [ ]:
#@title 7. Evaluate all models
for model in MODELS:
    for cat in CATEGORIES:
        print(f'Evaluating {cat}/{model}...', flush=True)
        subprocess.run([
            sys.executable, '-u', 'scripts/03_evaluate.py',
            '--category', cat,
            '--model', model,
        ])

In [ ]:
#@title 8. Generate paper plots
subprocess.run([sys.executable, '-u', 'scripts/04_generate_paper_plots.py'], check=True)
print('Paper plots generated.')

In [ ]:
#@title 9. Save results to Drive
if USE_DRIVE:
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    for subdir in ['output/results', 'output/models', 'output/figures', 'data/interim', 'data/processed/carbon', 'data/raw', 'docs']:
        src = PROJECT_ROOT / subdir
        dst = DRIVE_RESULTS / subdir
        if src.exists():
            dst.mkdir(parents=True, exist_ok=True)
            subprocess.run(['rsync', '-a', str(src) + '/', str(dst) + '/'], check=True)
            print(f'Synced {subdir} -> Drive')
    print(f'\nAll results saved to {DRIVE_RESULTS}')
else:
    print('Drive disabled. Results are in /content/carbon-aware-recsys/output/')

In [ ]:
#@title 10. Summary
import json

print(f'{"="*70}')
print('RESULTS SUMMARY')
print(f'{"="*70}\n')

for model in MODELS:
    for cat in CATEGORIES:
        metrics_path = PROJECT_ROOT / 'output' / 'results' / f'{cat}_{model}_reranking_metrics.json'
        if metrics_path.exists():
            with open(metrics_path) as f:
                data = json.load(f)
            s = data['summary']
            print(f'{cat}/{model}:')
            print(f"  Baseline NDCG@10={s.get('baseline_NDCG@10', 'N/A'):.4f}  carbon={s['baseline_carbon_kg']:.2f} kg")
            print(f"  Greenest  NDCG@10={s.get('greenest_NDCG@10', 'N/A'):.4f}  carbon={s['greenest_carbon_kg']:.2f} kg  (lambda={s['greenest_lambda']})")
            print(f"  Carbon reduction: {s['carbon_reduction_pct']:.1f}%\n")
        else:
            print(f'{cat}/{model}: NO RESULTS\n')